# Multimodal models: text, images and more

Exercise notebook for the lesson **[Multimodal models: text, images and more](https://ramkumarmd.com/medical-ai/generative-ai-llms/multimodal-models)** in the *Generative AI & LLMs* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Multimodal models process more than one kind of data, most often images and text. Vision-language models learn a shared space in which an image and a matching description lie close together, which allows zero-shot classification and draft report generation. They inherit the shortcut problem of imaging AI: they can learn hospital, scanner or label artefacts instead of disease. Evaluations of AI-drafted radiology reports find clinically significant errors in both AI and human reports, so their role so far is assistive.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Zero-shot classification in a shared space

Toy embeddings for three chest X-ray findings, with a scanner artefact visible in some images.

In [ ]:
import numpy as np

rng = np.random.default_rng(12)
dim = 32
classes = ["normal", "pneumonia", "pleural effusion"]
concepts = rng.normal(size=(3, dim))       # what each finding "looks like" in the shared space
hospital_tag = rng.normal(size=dim)        # a scanner/hospital artefact visible in images

def normalise(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# Text embeddings for prompts like "a chest X-ray showing pneumonia"
text = normalise(concepts + rng.normal(0, 0.2, (3, dim)))

def make_images(n, p_tag_by_class):
    labels = rng.integers(0, 3, n)
    tagged = rng.random(n) < np.array(p_tag_by_class)[labels]
    imgs = concepts[labels] + rng.normal(0, 3.0, (n, dim)) + 3.0 * tagged[:, None] * hospital_tag
    return normalise(imgs), labels

def zero_shot_accuracy(imgs, labels):
    predictions = (imgs @ text.T).argmax(axis=1)    # most similar text prompt
    return (predictions == labels).mean()

imgs, labels = make_images(3000, [0.1, 0.1, 0.1])
print(f"Zero-shot accuracy: {zero_shot_accuracy(imgs, labels):.1%}")

# Zero-shot accuracy: 77.8%

### Shortcuts: learning the hospital instead of the disease

Continuing from the previous block: train a classifier on embeddings at one hospital, test at a new one.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Development hospital: pneumonia patients mostly imaged on the tagged portable scanner
dev_imgs, dev_labels = make_images(3000, [0.1, 0.9, 0.1])
probe = LogisticRegression(max_iter=2000).fit(dev_imgs, dev_labels)
test_same, y_same = make_images(3000, [0.1, 0.9, 0.1])
test_new, y_new = make_images(3000, [0.5, 0.5, 0.5])   # new hospital: scanner unrelated to diagnosis
print(f"Fine-tuned classifier, same hospital: {probe.score(test_same, y_same):.1%}")
print(f"Fine-tuned classifier, new hospital:  {probe.score(test_new, y_new):.1%}")
print(f"Zero-shot, new hospital:              {zero_shot_accuracy(test_new, y_new):.1%}")

# Fine-tuned classifier, same hospital: 84.7%
# Fine-tuned classifier, new hospital:  67.6%
# Zero-shot, new hospital:              72.6%

## Exercise

**Goal:** Explore zero-shot classification and shortcut learning with toy embeddings, then with an open chest X-ray dataset if available.

**Data:** Toy embeddings generated in the notebook; optional extension uses an openly licensed chest X-ray dataset (Generated in the notebook; extension dataset under its own licence)

### Step 1

Reproduce the zero-shot and shortcut results, and vary how strongly the artefact is tied to pneumonia.

In [ ]:
# Your code here


### Step 2

Remove the artefact direction from the embeddings (project it out) and retrain; how does new-hospital accuracy change?

In [ ]:
# Your code here


### Step 3

Try 'prompt ensembling': average several text descriptions per class and compare zero-shot accuracy.

In [ ]:
# Your code here


### Step 4

Optional: with an open chest X-ray dataset and an open vision-language model, compare zero-shot accuracy for two sites or acquisition types.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/multimodal-models](https://ramkumarmd.com/medical-ai/generative-ai-llms/multimodal-models). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.